# TrafficVision — YOLOv8 Vehicle Detection Training

Upload **`dataset.zip`** and (**optionally**) **`yolov8n.pt`** to this Colab folder, then run all cells.

Training: **25 epochs**, 416 px, GPU if available (else CPU). Class order: `0=bus, 1=car, 2=truck, 3=van`.

At the end, `vehicle_detector.onnx`, `class_names.json`, `training_summary.json` and `best.pt` are downloaded. Send those to me to finish the app and deployment.

In [ ]:
!pip install -q ultralytics onnx onnxsim opencv-python pandas
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
from pathlib import Path
import zipfile

if not Path('dataset/train/images').exists():
    with zipfile.ZipFile('dataset.zip') as zf:
        zf.extractall('.')
print('Train images:', len(list(Path('dataset/train/images').glob('*.jpg'))))
print('Val images:', len(list(Path('dataset/val/images').glob('*.jpg'))))

In [ ]:
from pathlib import Path

root = Path('dataset').resolve().as_posix()
Path('data.yaml').write_text(
    f'path: {root}\ntrain: train/images\nval: val/images\nnc: 4\nnames: [bus, car, truck, van]\n',
    encoding='utf-8',
)
print(Path('data.yaml').read_text())

In [ ]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else 'cpu'
batch = 16 if device != 'cpu' else 8
print(f'device={device}, batch={batch}')

model = YOLO('yolov8n.pt')
model.train(
    data='data.yaml', epochs=25, imgsz=416, batch=batch,
    device=device, workers=0, patience=0,
    project='runs', name='vehicle_detector', exist_ok=True,
)

In [ ]:
import json, shutil
from pathlib import Path

metrics = model.val(data='data.yaml', split='val', imgsz=416, batch=batch, device=device, workers=0)
summary = {
    'map50_95': round(float(metrics.box.map), 4),
    'map50': round(float(metrics.box.map50), 4),
    'per_class_map50_95': [round(float(x), 4) for x in metrics.box.maps],
    'epochs': 25,
    'imgsz': 416,
    'dataset': 'Roboflow UA-DETRAC CCTV vehicles (YOLOv8 export, 4 classes)',
}
Path('training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

onnx_file = model.export(format='onnx', imgsz=416, dynamic=False, opset=17)
shutil.copy2(onnx_file, 'vehicle_detector.onnx')
Path('class_names.json').write_text(
    json.dumps({str(i): name for i, name in model.names.items()}, indent=2), encoding='utf-8'
)
print('Saved vehicle_detector.onnx, class_names.json, training_summary.json')

In [ ]:
try:
    from google.colab import files
    from pathlib import Path
    for name in ['training_summary.json', 'class_names.json', 'vehicle_detector.onnx']:
        files.download(name)
    checkpoints = sorted(Path('.').rglob('best.pt'))
    if checkpoints:
        print('Found checkpoint:', checkpoints[-1])
        files.download(str(checkpoints[-1]))
    else:
        print('best.pt not found; not needed for the Streamlit app. Skipping.')
except ImportError:
    print('Not running in Colab; output files are in the current directory.')